In [1]:
#STEP-01 Loading all libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split # spliting data into training and testing sets
from sklearn.ensemble import RandomForestClassifier, IsolationForest # Random Forest classifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score # Check accuracy score 

In [2]:
#STEP-02
# Loading dataset & display the first few rows of the dataframe
df = pd.read_csv('BAIoT_mirai.csv')

In [3]:
# Assuming our labels are in a column named 'multilabel' cosidering normal as 0 and rest anomalous as 1
#df['project'] = df['project'].apply(lambda x: 0 if x == 'BRCA' else 1)
df.shape

(75164, 116)

In [4]:
df.columns

Index(['MI_dir_L5_weight', 'MI_dir_L5_mean', 'MI_dir_L5_variance',
       'MI_dir_L3_weight', 'MI_dir_L3_mean', 'MI_dir_L3_variance',
       'MI_dir_L1_weight', 'MI_dir_L1_mean', 'MI_dir_L1_variance',
       'MI_dir_L0.1_weight',
       ...
       'HpHp_L0.1_covariance', 'HpHp_L0.1_pcc', 'HpHp_L0.01_weight',
       'HpHp_L0.01_mean', 'HpHp_L0.01_std', 'HpHp_L0.01_magnitude',
       'HpHp_L0.01_radius', 'HpHp_L0.01_covariance', 'HpHp_L0.01_pcc',
       'multilabel'],
      dtype='object', length=116)

In [5]:
df.fillna(0, inplace=True)

df_y=df['multilabel']

df_x=df.drop(labels=['multilabel'], axis=1)

In [6]:
pd.options.display.max_rows = 100
df_x.dtypes

MI_dir_L5_weight         float64
MI_dir_L5_mean           float64
MI_dir_L5_variance       float64
MI_dir_L3_weight         float64
MI_dir_L3_mean           float64
                          ...   
HpHp_L0.01_std           float64
HpHp_L0.01_magnitude     float64
HpHp_L0.01_radius        float64
HpHp_L0.01_covariance    float64
HpHp_L0.01_pcc           float64
Length: 115, dtype: object

In [7]:
df_y=pd.DataFrame(df_y)
df_y

,multilabel
0,normal
1,mirai-scan
2,mirai-udpplain
3,normal
4,normal
...,...
75159,normal
75160,normal
75161,normal
75162,mirai-udp


In [8]:
df_y.loc[df_y['multilabel'] == 'normal', 'multilabel'] = 0
df_y.loc[df_y['multilabel'] != 0, 'multilabel'] = 1
df_y['multilabel']=df_y['multilabel'].astype(int)

In [9]:
df_y

,multilabel
0,0
1,1
2,1
3,0
4,0
...,...
75159,0
75160,0
75161,0
75162,1


In [10]:
#find categorical data
df_x.nunique()
#maybe categorical: Protocol;Fwd PSH Flags; Bwd PSH Flags; URG Flags; Bwd URG Flags
#ignoring this for now

MI_dir_L5_weight         48565
MI_dir_L5_mean           31324
MI_dir_L5_variance       37188
MI_dir_L3_weight         55012
MI_dir_L3_mean           42232
                         ...  
HpHp_L0.01_std             727
HpHp_L0.01_magnitude       448
HpHp_L0.01_radius          729
HpHp_L0.01_covariance     6534
HpHp_L0.01_pcc            5281
Length: 115, dtype: int64

In [11]:
#moving to numpy as it is easier to check columns
np_x=df_x.to_numpy()
identical=np.all(np_x == np_x[0,:], axis = 0)
identical #True means there are some columns that must be dropped

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False])

In [12]:
to_be_deleted=np.where(identical==True)
to_be_deleted

(array([], dtype=int64),)

In [13]:
#deleting column, back to pandas as it is easier
data_pd=pd.DataFrame(np_x)

In [14]:
data_pd.drop(data_pd.columns[to_be_deleted[0]], inplace=True, axis=1)

In [15]:
#back to numpy and normalize
np_x=data_pd.to_numpy()
# normalize each column by subtracting its mean and dividing by its standard deviation
col_means = np.mean(np_x, axis=0)

data_normalized = (np_x - col_means) / np.std(np_x, axis=0)

In [16]:
# STEP-04: 
# Spliting data into training and testing sets

# Define X and y
X = df[['MI_dir_L5_weight',
 'MI_dir_L5_mean',
 'MI_dir_L5_variance',
 'MI_dir_L3_weight',
 'MI_dir_L3_mean',
 'MI_dir_L3_variance',
 'MI_dir_L1_weight',
 'MI_dir_L1_mean',
 'MI_dir_L1_variance',
 'MI_dir_L0.1_weight',
 'MI_dir_L0.1_mean',
 'MI_dir_L0.1_variance',
 'MI_dir_L0.01_weight',
 'MI_dir_L0.01_mean',
 'MI_dir_L0.01_variance',
 'H_L5_weight',
 'H_L5_mean',
 'H_L5_variance',
 'H_L3_weight',
 'H_L3_mean',
 'H_L3_variance',
 'H_L1_weight',
 'H_L1_mean',
 'H_L1_variance',
 'H_L0.1_weight',
 'H_L0.1_mean',
 'H_L0.1_variance',
 'H_L0.01_weight',
 'H_L0.01_mean',
 'H_L0.01_variance',
 'HH_L5_weight',
 'HH_L5_mean',
 'HH_L5_std',
 'HH_L5_magnitude',
 'HH_L5_radius',
 'HH_L5_covariance',
 'HH_L5_pcc',
 'HH_L3_weight',
 'HH_L3_mean',
 'HH_L3_std',
 'HH_L3_magnitude',
 'HH_L3_radius',
 'HH_L3_covariance',
 'HH_L3_pcc',
 'HH_L1_weight',
 'HH_L1_mean',
 'HH_L1_std',
 'HH_L1_magnitude',
 'HH_L1_radius',
 'HH_L1_covariance',
 'HH_L1_pcc',
 'HH_L0.1_weight',
 'HH_L0.1_mean',
 'HH_L0.1_std',
 'HH_L0.1_magnitude',
 'HH_L0.1_radius',
 'HH_L0.1_covariance',
 'HH_L0.1_pcc',
 'HH_L0.01_weight',
 'HH_L0.01_mean',
 'HH_L0.01_std',
 'HH_L0.01_magnitude',
 'HH_L0.01_radius',
 'HH_L0.01_covariance',
 'HH_L0.01_pcc',
 'HH_jit_L5_weight',
 'HH_jit_L5_mean',
 'HH_jit_L5_variance',
 'HH_jit_L3_weight',
 'HH_jit_L3_mean',
 'HH_jit_L3_variance',
 'HH_jit_L1_weight',
 'HH_jit_L1_mean',
 'HH_jit_L1_variance',
 'HH_jit_L0.1_weight',
 'HH_jit_L0.1_mean',
 'HH_jit_L0.1_variance',
 'HH_jit_L0.01_weight',
 'HH_jit_L0.01_mean',
 'HH_jit_L0.01_variance',
 'HpHp_L5_weight',
 'HpHp_L5_mean',
 'HpHp_L5_std',
 'HpHp_L5_magnitude',
 'HpHp_L5_radius',
 'HpHp_L5_covariance',
 'HpHp_L5_pcc',
 'HpHp_L3_weight',
 'HpHp_L3_mean',
 'HpHp_L3_std',
 'HpHp_L3_magnitude',
 'HpHp_L3_radius',
 'HpHp_L3_covariance',
 'HpHp_L3_pcc',
 'HpHp_L1_weight',
 'HpHp_L1_mean',
 'HpHp_L1_std',
 'HpHp_L1_magnitude',
 'HpHp_L1_radius',
 'HpHp_L1_covariance',
 'HpHp_L1_pcc',
 'HpHp_L0.1_weight',
 'HpHp_L0.1_mean',
 'HpHp_L0.1_std',
 'HpHp_L0.1_magnitude',
 'HpHp_L0.1_radius',
 'HpHp_L0.1_covariance',
 'HpHp_L0.1_pcc',
 'HpHp_L0.01_weight',
 'HpHp_L0.01_mean',
 'HpHp_L0.01_std',
 'HpHp_L0.01_magnitude',
 'HpHp_L0.01_radius',
 'HpHp_L0.01_covariance',
 'HpHp_L0.01_pcc']]  # Features
y = df['multilabel']  # Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state = 42)

In [17]:
# STEP-04(B)
# Random Forest Classifier model with default parameters

# instantiate the classifier 
rfc = RandomForestClassifier(random_state=0)

# fit the model
rfc.fit(X_train, y_train)

# Predict the Test set results
y_pred = rfc.predict(X_test)

# Find important features with Random Forest model
# create the classifier with n_estimators = 100
clf = RandomForestClassifier(n_estimators=100, random_state=0)

# fit the model to the training set
clf.fit(X_train, y_train)

RandomForestClassifier(random_state=0)

In [18]:
# STEP-06
# Performance Metrices
y_pred_train = rfc.predict(X_train)
y_pred_test = rfc.predict(X_test)

# View accuracy score 
# Sprint('Model accuracy score : {0:0.4f}'. format(accuracy_score(y_test, y_pred)))
# Calculate metrics for the training set
accuracy_train = accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train, average='weighted')
recall_train = recall_score(y_train, y_pred_train, average='weighted')
f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Calculate metrics for the testing set
accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test, average='weighted')
recall_test = recall_score(y_test, y_pred_test, average='weighted')
f1_test = f1_score(y_test, y_pred_test, average='weighted')

print(f'Training Accuracy: {accuracy_train}')
print(f'Training Precision: {precision_train}')
print(f'Training Recall: {recall_train}')
print(f'Training F1-score: {f1_train}')
print()

print(f'Testing Accuracy: {accuracy_test}')
print(f'Testing Precision: {precision_test}')
print(f'Testing Recall: {recall_test}')
print(f'Testing F1-score: {f1_test}')

Training Accuracy: 1.0
Training Precision: 1.0
Training Recall: 1.0
Training F1-score: 1.0

Testing Accuracy: 1.0
Testing Precision: 1.0
Testing Recall: 1.0
Testing F1-score: 1.0


In [19]:
# Check for missing values and handle them appropriately
if X.isnull().sum().any() or y.isnull().sum().any():
    X = X.fillna(X.mean())  # Example: filling missing values with the mean
    y = y.fillna(y.mode()[0])  # Example: filling missing values with the mode  

# Convert y to the format expected by IsolationForest if needed
y = y.apply(lambda x: -1 if x == 0 else 1)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [20]:
#STEP-04
# Fit the Isolation Forest model on the training data
model = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
model.fit(X_train)
model.fit(X_test)

C:\Users\Setha\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
C:\Users\Setha\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


IsolationForest(contamination=0.1, random_state=42)

In [21]:
#STEP-05
# Predict anomalies on the training data
y_train_pred = model.predict(X_train)

# Step 6: Predict anomalies on the testing data
y_test_pred = model.predict(X_test)

# Step 7: Calculate accuracy and other metrics for training data
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

# Step 8: Calculate accuracy and other metrics for testing data
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

# Print the results for training data
# print('Model accuracy score : {0:0.4f}'. format(accuracy_score(y_test, y_test_pred)))
print('Training Scores:')
print(f'Accuracy: {train_accuracy}')
print(f'Precision: {train_precision}')
print(f'Recall: {train_recall}')
print(f'F1 Score: {train_f1}')
print()

# Print the results for testing data
print('Testing Scores:')
print(f'Accuracy: {test_accuracy}')
print(f'Precision: {test_precision}')
print(f'Recall: {test_recall}')
print(f'F1 Score: {test_f1}')

Training Scores:
Accuracy: 0.9055384498422473
Precision: 1.0
Recall: 0.9055384498422473
F1 Score: 0.9504278960282471

Testing Scores:
Accuracy: 0.9
Precision: 1.0
Recall: 0.9
F1 Score: 0.9473684210526316
